In [35]:
# PUTTING IT ALL TOGETHER

In [36]:
# Install Google ADK for Python
%pip install google-adk --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [37]:
# Install LangChain community and Tavily client for the search tool
%pip install langchain_community tavily-python --quiet


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [38]:
# (optional) Verify installation
%pip show google-adk
%pip show langchain_community 
%pip show tavily-python

Name: google-adk
Version: 1.0.0
Summary: Agent Development Kit
Home-page: https://google.github.io/adk-docs/
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: authlib, click, fastapi, google-api-python-client, google-cloud-aiplatform, google-cloud-secret-manager, google-cloud-speech, google-cloud-storage, google-genai, graphviz, mcp, opentelemetry-api, opentelemetry-exporter-gcp-trace, opentelemetry-sdk, pydantic, python-dotenv, PyYAML, sqlalchemy, tzlocal, uvicorn
Required-by: 
Note: you may need to restart the kernel to use updated packages.
Name: langchain-community
Version: 0.3.24
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /home/codespace/.python/current/lib/python3.12/site-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, reque

In [39]:
# Configure environment
import os
# We use Gemini as our language model and ensure we are not using Vertex AI for this demo.
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

In [40]:
# Import Required Modules
import requests
from typing import Dict, Any, Optional
from datetime import datetime, timedelta
from collections import Counter
from dateutil import parser as date_parser
import calendar
import re

from google.adk.agents import Agent, ParallelAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import google_search
from google.genai import types
from IPython.display import Markdown, display

In [41]:
# Tavily Search API: Set Up
# API Key Management: Always use environment variables or a secure vault for credentials!
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError(
        "TAVILY_API_KEY environment variable not set. "
        "Never hardcode API keys in your code. "
        "Set it securely in your environment or use a secrets manager."
)

In [42]:
# Tavily LangChain tool: Initilaization
# This tool allows the agent to perform live web searches using Tavily.
from langchain_community.tools import TavilySearchResults
tavily_tool_instance = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    include_answer=True,
    include_raw_content=True,
    include_images=True,
)

# Wrap it for ADK compatibility
from google.adk.tools.langchain_tool import LangchainTool
adk_tavily_tool = LangchainTool(tool=tavily_tool_instance)

In [43]:
# OpenWeather API: Set Up
OPEN_WEATHER_API_KEY = os.environ.get("OPEN_WEATHER_API_KEY")
if not OPEN_WEATHER_API_KEY:
    raise ValueError("OPEN_WEATHER_API_KEY environment variable not set.  Please set it.")

In [44]:
# Helper Functions: Parse flexible date expressions for weather queries
def parse_flexible_date_range(date_str: str) -> Optional[tuple]:
    """
    Parses a wide variety of date expressions and returns (start_date, end_date).
    Supports:
    - 'today', 'tomorrow', 'in 3 days'
    - 'this weekend', 'next weekend'
    - 'next week', 'this week'
    - 'first week of July', 'second week of August', etc.
    - Specific dates: '2025-07-01', 'July 1', '07-01'
    Returns (datetime, datetime) or None if parsing fails.
    """
    date_str = date_str.lower().strip()
    today = datetime.today()
    weekday = today.weekday()  # Monday=0, Sunday=6

    # Today, tomorrow, in X days
    target_date = None
    if date_str in ["today", "now"]:
        target_date = today
    elif date_str == "tomorrow":
        target_date = today + timedelta(days=1)
    
    match = re.match(r"in (\\d+) days?", date_str)
    if match:
        days = int(match.group(1))
        target_date = today + timedelta(days=days)

    if target_date:
        start_of_day = target_date.replace(hour=0, minute=0, second=0, microsecond=0)
        end_of_day = target_date.replace(hour=23, minute=59, second=59, microsecond=999999)
        return start_of_day, end_of_day
    
    match = re.match(r"(?:next|for the next) (\d+) days?", date_str)
    if match:
        days = int(match.group(1))
        start = today
        end = today + timedelta(days=days - 1)
        return start, end

    # This week (Monday to Sunday)
    if date_str == "this week":
        start = today - timedelta(days=weekday)
        end = start + timedelta(days=6)
        return start, end

    # Next week (Monday to Sunday)
    if date_str == "next week":
        start = today - timedelta(days=weekday) + timedelta(days=7)
        end = start + timedelta(days=6)
        return start, end

    # This weekend (Saturday & Sunday)
    if date_str == "this weekend":
        saturday = today + timedelta((5 - weekday) % 7)
        sunday = saturday + timedelta(days=1)
        return saturday, sunday

    # Next weekend (Saturday & Sunday)
    if date_str == "next weekend":
        saturday = today + timedelta((5 - weekday) % 7 + 7)
        sunday = saturday + timedelta(days=1)
        return saturday, sunday

    # "first week of July", "second week of August", etc.
    match = re.match(r"(first|second|third|fourth|last) week of (\w+)", date_str)
    if match:
        week_map = {
            "first": 0, "second": 1, "third": 2, "fourth": 3, "last": -1
        }
        week_num = week_map[match.group(1)]
        month_str = match.group(2)
        try:
            month = list(calendar.month_name).index(month_str.capitalize())
            year = today.year
            # If the month has already passed, assume next year
            if month < today.month:
                year += 1
            cal = calendar.monthcalendar(year, month)
            if week_num == -1:
                week = cal[-1]
            else:
                week = cal[week_num]
            # Get Monday (0) and Sunday (6) of that week
            start_day = week[0] if week[0] != 0 else week[calendar.SATURDAY]
            end_day = week[6] if week[6] != 0 else week[calendar.FRIDAY]
            start = datetime(year, month, start_day)
            # Find first non-zero day for start, last non-zero for end
            for d in week:
                if d != 0:
                    start = datetime(year, month, d)
                    break
            for d in reversed(week):
                if d != 0:
                    end = datetime(year, month, d)
                    break
            return start, end
        except Exception:
            return None

    # Try parsing as a specific date or date range
    try:
        # "July 1", "07-01", "2025-07-01"
        dt = date_parser.parse(date_str, fuzzy=True, default=today)
        return dt, dt
    except Exception:
        pass

    # "July 1 to July 5", "07-01 to 07-05"
    if " to " in date_str:
        parts = date_str.split(" to ")
        try:
            start = date_parser.parse(parts[0], fuzzy=True, default=today)
            end = date_parser.parse(parts[1], fuzzy=True, default=today)
            return start, end
        except Exception:
            return None

    return None

In [45]:
# Custom Tool: Get Current weather (stateful version)
def get_current_weather_from_openweather_stateful(city: str, tool_context) -> Dict[str, Any]:
    preferred_unit = tool_context.state.get("user_preference_temperature_unit", "metric")
    unit_symbol = "°C" if preferred_unit == "metric" else "°F"
    if not city:
        city = tool_context.state.get("last_city_checked_stateful")
    try:
        url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPEN_WEATHER_API_KEY}&units={preferred_unit}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        if data["cod"] == 200:
            description = data["weather"][0]["description"]
            temperature = data["main"]["temp"]
            humidity = data["main"]["humidity"]
            wind_speed = data["wind"]["speed"]
            report = (
                f"The weather in {city} is {description} with a temperature of {temperature}{unit_symbol}. "
                f"Humidity is {humidity}% and wind speed is {wind_speed} m/s."
            )
            tool_context.state["last_city_checked_stateful"] = city
            return {"status": "success", "report": report}
        else:
            return {"status": "error", "error_message": f"OpenWeatherMap API Error: {data['message']}"}
    except requests.exceptions.RequestException as e:
        return {"status": "error", "error_message": f"Error fetching weather data: {e}"}
    except KeyError as e:
        return {"status": "error", "error_message": f"Error parsing weather data: Missing key: {e}"}
    except Exception as e:
        return {"status": "error", "error_message": f"An unexpected error occurred: {e}"}

In [46]:
# Custom Tool: Get weather summary for a flexible date range (stateful version)
def get_weather_summary_from_openweather_stateful(
    city: Optional[str], date_expr: Optional[str], tool_context
) -> Dict[str, Any]:
    if not city:
        city = tool_context.state.get("last_city_checked_stateful")
    if not city:
        return {"status": "error", "error_message": "No city specified or remembered."}
    if not date_expr:
        date_expr = tool_context.state.get("last_date_expr_stateful")
    if not date_expr:
        return {"status": "error", "error_message": "No date or date range specified or remembered."}

    preferred_unit = tool_context.state.get("user_preference_temperature_unit", "metric")
    unit_symbol = "°C" if preferred_unit == "metric" else "°F"
    date_range = parse_flexible_date_range(date_expr)
    if not date_range:
        return {"status": "error", "error_message": f"Could not parse date expression: '{date_expr}'"}
    start, end = date_range
    num_days = (end - start).days + 1

    print(start, end, num_days)  # Debugging line to check parsed date range

    try:
        url = f"http://api.openweathermap.org/data/2.5/forecast?q={city}&appid={OPEN_WEATHER_API_KEY}&units={preferred_unit}"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        current_url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPEN_WEATHER_API_KEY}&units={preferred_unit}"
        current_response = requests.get(current_url)
        current_response.raise_for_status()
        current_data = current_response.json()
        all_temps = []
        all_descriptions = []
        for entry in data.get("list", []):
            entry_date = datetime.fromtimestamp(entry["dt"])
            if start <= entry_date <= end:
                all_temps.append(entry["main"]["temp"])
                all_descriptions.append(entry["weather"][0]["description"])
        if not all_temps:
            all_temps = [current_data["main"]["temp"]]
            all_descriptions = [current_data["weather"][0]["description"]]
        print(all_temps)  # Debugging line to check collected temperatures
        min_temp = round(min(all_temps), 1)
        max_temp = round(max(all_temps), 1)
        avg_temp = round(sum(all_temps) / len(all_temps), 1)
        weather_types = [desc for desc, _ in Counter(all_descriptions).most_common(3)]
        weather_summary = {
            "destination": data["city"]["name"] if "city" in data else city,
            "country": data["city"]["country"] if "city" in data else "",
            "trip_length": num_days,
            "date_range": f"{start.strftime('%Y-%m-%d')} to {end.strftime('%Y-%m-%d')}",
            "avg_temp": avg_temp,
            "temp_range": f"{min_temp}{unit_symbol} to {max_temp}{unit_symbol}",
            "weather_types": weather_types
        }
        tool_context.state["last_city_checked_stateful"] = city
        tool_context.state["last_date_expr_stateful"] = date_expr
        return {"status": "success", "summary": weather_summary}
    except requests.exceptions.RequestException as e:
        return {"status": "error", "error_message": f"Error fetching weather data: {e}"}
    except KeyError as e:
        return {"status": "error", "error_message": f"Error parsing weather data: Missing key: {e}"}
    except Exception as e:
        return {"status": "error", "error_message": f"An unexpected error occurred: {e}"}


In [47]:
# Weather Agent: Definition
weather_agent = Agent(
    name="weather_agent_stateful",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Provides current weather and weather summaries, using user preferences and context.",
    instruction=(
        "You are a helpful weather assistant. "
        "If the user does not specify a city, use the last city checked from session state. "
        "If the user does not specify a date, use the last date range from session state. "
        "Accept natural language date expressions like 'this weekend' or 'next week'. "
        "When the user asks for the current weather in a city, "
        "use the 'get_current_weather_from_openweather_stateful' tool. "
        "When the user asks for weather for a date or date range, "
        "use the 'get_weather_summary_from_openweather_stateful' tool. "
        "If a tool returns an error, inform the user politely. "
    ),
    tools=[get_current_weather_from_openweather_stateful, get_weather_summary_from_openweather_stateful],
    output_key="weather_summary"
)

In [48]:
# Itinerary Agent: Definition
itinerary_agent = Agent(
    name="itinerary_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="An agent that creates a travel itinerary for a given destination and trip duration, using Google Search for up-to-date recommendations.",
    instruction="""
        You are a helpful and creative travel itinerary planner.
        When the user provides a destination and trip duration, use the [google_search] tool to find current and popular attractions, restaurants, and activities for each day.
        For each day, create a detailed schedule with 2-4 activities, including at least one meal suggestion and one local attraction, prioritizing the user's interests (such as art and food).
        Present the itinerary in a clear, easy-to-read markdown format, organized by day.
        If you need more information, ask the user for preferences or interests, but if you have enough, proceed to generate a full itinerary.
        Example format:

        ### Day 1
        - Morning: [Attraction or activity]
        - Lunch: [Restaurant or food experience]
        - Afternoon: [Attraction or activity]
        - Evening: [Dinner suggestion or event]

        ### Day 2
        ...

        Always use the [google_search] tool to find the latest recommendations for each activity or restaurant.
        Be concise, friendly, and ensure the itinerary is complete for all days requested.
    """,
    tools=[google_search],
    output_key="itinerary"
)

In [49]:
# Packing List Agent: Definition
packing_list_agent = Agent(
    name="packing_list_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Generates a packing list based on itinerary, destination, and user preferences, including weather information.", # Added weather
    instruction="""
        You are a helpful travel assistant.
        The user will provide their destination, trip duration, and a detailed itinerary (including planned activities, locations, and types of events for each day).
        You will also get the weather summary.
        Your task is to generate a complete, practical packing list tailored to the specific activities, locations in the itinerary, and the weather forecast.

        Itinerary: {itinerary}
        Weather Summary: {weather_summary}

        Use the provided itinerary and weather summary to determine what items are necessary for each day and activity.
        Organize the packing list by category (such as Clothing, Toiletries, Electronics, Documents, and Other Essentials).
        If you notice a special activity or requirement in the itinerary, include specific items for it.
        If any information is missing, make reasonable assumptions based on the itinerary, destination, and weather forecast—do not ask the user for more details unless absolutely necessary.
        Present the packing list in a clear, easy-to-read markdown format. For example:

        #### Clothing
        - Comfortable walking shoes
        - Lightweight jacket
        - Dress for dinner at a nice restaurant
        - Swimwear (if beach or pool in itinerary)
        - ...

        #### Toiletries
        - Toothbrush and toothpaste
        - Sunscreen
        - ...

        #### Electronics
        - Phone and charger
        - Plug adapter (if needed)
        - ...

        #### Documents
        - Passport
        - Travel insurance
        - ...

        #### Other Essentials
        - Daypack for city tours
        - Water bottle
        - ...

        Only ask clarifying questions if the itinerary is extremely vague or unclear. Otherwise, use your best judgment and generate a full, actionable packing list.
        """,
    output_key="packing_list"
)

In [50]:
# Latest Events Agent: Definition
# Define the agent with Tavily as a third-party tool
# The instruction set is the heart of the agent. Make it specific, structured, and safe.
latest_events_agent = Agent(
    name="latest_events_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="An agent that uses the Tavily search tool to find and summarize the latest events, festivals, and activities for a given destination, providing up-to-date and user-friendly travelinformation.",
    instruction="""
        You are a travel assistant.
        When a user asks about current or upcoming events, festivals, or activities at a destination, use the Tavily search tool to find the most recent and relevant information.
        Summarize your findings in a clear, concise, and user-friendly way.
        Only use the Tavily tool for event or activity-related queries, and ensure your answers are accurate and helpful.
        If you cannot find relevant information, politely let the user know.
        Always provide the source of your information.
        If the user asks about general travel advice or information not related to current events, politely redirect them to other resources.
        Never expose API keys or sensitive data in your responses.
    """,
    tools=[adk_tavily_tool],
    output_key="latest_events"
)

In [51]:
# Create a parallel agent to run the first two tasks concurrently
gather_info_agent = ParallelAgent(
    name="gather_info_agent",
    sub_agents=[latest_events_agent, itinerary_agent] # Run latest_events and itinerary in parallel
)

In [52]:
# Personalizer Agent: Definition
personalizer_agent = Agent(
    name="personalizer_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Refines a travel itinerary by integrating relevant, timely events based on user interests.",
    instruction="""
        You are a travel personalization expert. Your task is to review a generated itinerary and a list of current events, and then create a final, unified plan.

        You will receive a pre-generated itinerary and a list of events. Your goal is to see if any of the events from the list can be integrated into the itinerary to make it more personalized and exciting.

        **Instructions:**
        1.  Analyze the itinerary and the list of events.
        2.  Identify events that align with the user's interests (like art, food, music) mentioned in the itinerary.
        3.  If a relevant event's dates overlap with the trip dates, suggest integrating it into the itinerary. You can replace a more generic activity with the specific event.
        4.  Present a single, final, polished itinerary. If you make a change, briefly mention why (e.g., "I've replaced the museum visit with the special 'Shiko Munakata' exhibition, as it aligns with your interest in art.").
        5.  If no events can be reasonably integrated, simply present the original itinerary as the final plan.

        **Input Itinerary:**
        {itinerary}

        **Input Events List:**
        {latest_events}
    """,
    output_key="personalized_itinerary"
)

In [53]:
# Packing and Outfit Suggestion Agent (New Agent)
# This Agent leverages the itinerary, weather, and generates a packing list
# and suggests outfits.
packing_and_outfit_agent = Agent(
    name="packing_and_outfit_agent",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Generates a packing list and suggests daily outfits based on the itinerary and weather forecast.",
    instruction="""
        You are a helpful travel assistant.  You will receive a detailed itinerary and weather information for the trip.

        Itinerary: {itinerary}
        Weather Summary: {weather_summary}

        Based on this information:
        1.  Generate a packing list, categorized by type (Clothing, Toiletries, Electronics, Documents, Other).  Consider the activities in the itinerary and the weather summary.
        2.  For each day of the itinerary, suggest a possible outfit based on the day's planned activities and the expected weather.
        Present the packing list and outfit suggestions in a clear, easy-to-read Markdown format.

        Example:

        #### Packing List

        #### Clothing
        - Comfortable walking shoes
        - Light jacket
        - Dress for dinner
        - ...

        #### Toiletries
        - Toothbrush
        - Sunscreen
        - ...

        #### Day 1: [Day's activity]  (Weather: [Weather description])
        - Outfit: [Detailed description]

        #### Day 2: ...
        - Outfit: ...
    """,
    output_key="packing_list_and_outfits"
)

In [54]:
# Final Pipeline - Orchestration (from 04_02 & Combining)

# Create the sequential pipeline
travel_pipeline_agent = SequentialAgent(
    name="travel_pipeline_agent",
    sub_agents=[
        weather_agent, # Gets weather, output is "weather_summary"
        gather_info_agent, # Parallel: itinerary & events
        personalizer_agent, # Personalize the itinerary
        packing_list_agent  # Packing list and outfit suggestions - NOW uses weather_summary!
    ],
    description="A travel assistant that gathers info, personalizes, and creates a packing list."
)

In [55]:
# Set up the session and user input
APP_NAME = "wanderwise_workflow_demo"
USER_ID = "user_001"
SESSION_ID = "workflow_session_001"

# Initialize the session service (from 04_05)
session_service = InMemorySessionService()
# Initialize the memory service
memory_service = InMemoryMemoryService()

# Create session using the correct method signature (from 04_05)
initial_state = {
    "user_preference_temperature_unit": "imperial"
}

# Create session using the correct method signature
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    state=initial_state
)


# Initialize the runner (from 04_05 & 02_04)
runner = Runner(
    agent=travel_pipeline_agent, # Use the combined agent
    app_name=APP_NAME,
    session_service=session_service,
    memory_service=memory_service
)

In [56]:
# Helper function to run and display agent responses (from 04_05 & 02_04 - Modified)
async def ask_agent(runner, prompt):  # Changed to a generic name
    content = types.Content(role="user", parts=[types.Part(text=prompt)])
    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response():
            # Extract only the text parts
            text_parts = [part.text for part in event.content.parts if hasattr(part, 'text') and part.text]
            # Join all text parts (in case there are multiple)
            text_response = "\n".join(text_parts)
            display(Markdown(f"**Prompt:** {prompt}\n\n---\n\n**Agent Response:**\n\n{text_response}"))

# Helper Function: To show session state (from 04_05)
async def show_context():
    # Get the latest session state
    current_session = await session_service.get_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )
    print("Session state (per conversation):")
    for k, v in current_session.state.items():
        print(f"  {k}: {v}")

# Helper function to update session state properly (from 04_05 - if you use it, otherwise remove)
async def update_temperature_unit(new_unit):
    """Update the temperature unit preference in the session state"""
    current_session = session_service.sessions[APP_NAME][USER_ID][SESSION_ID]
    current_session.state["user_preference_temperature_unit"] = new_unit
    print(f"Updated temperature unit to: {new_unit}")

In [59]:
# Main interaction flow

USER_INPUT = "I'm traveling to London for next 3 days and I love history, cocktails, and some fashion." # Start with user prompt.

await ask_agent(runner, USER_INPUT)
await show_context()

2025-06-08 18:09:31.014878 2025-06-10 18:09:31.014878 3
[61.18, 56.71, 50.79, 56.12, 62.31, 63.54, 64.63, 64.56, 59, 56.97, 54.52, 55.74, 60.8, 68.7, 68.7, 66.33]


**Prompt:** I'm traveling to London for next 3 days and I love history, cocktails, and some fashion.

---

**Agent Response:**

OK. The weather in London for the next 3 days will be partly cloudy with average temperature of 60.7F, with a low of 50.8F and a high of 68.7F.


**Prompt:** I'm traveling to London for next 3 days and I love history, cocktails, and some fashion.

---

**Agent Response:**

OK. Considering the partly cloudy weather, your interest in history, cocktails, and fashion, here's an updated 3-day itinerary for London from June 9 to June 11, 2025:

### Day 1 (June 9, 2025)
- Morning: Tower of London. Explore the historic castle, see the Crown Jewels, and learn about its history.
- Lunch: The Perkin Reveller near the Tower of London offers a modern British cuisine with historical charm.
- Afternoon: Vintage Shopping at Brick Lane. Explore the vintage fashion scene at Brick Lane.
- Evening: Cocktail Hour at The Churchill Bar & Terrace. Enjoy a historical theme and sophisticated cocktails. Followed by dinner at a nearby restaurant.

### Day 2 (June 10, 2025)
- Morning: British Museum. Focus on historical exhibits like the Rosetta Stone and Elgin Marbles.
- Lunch: Enjoy food near the British Museum.
- Afternoon: Fashion History at the Victoria and Albert Museum. Explore fashion through the ages at the V&A.
- Evening: Cocktail at The Savoy's American Bar. Experience a classic cocktail bar known for its history. Dinner at Kaspar's at The Savoy.

### Day 3 (June 11, 2025)
- Morning: Westminster Abbey & Houses of Parliament. Tour Westminster Abbey and visit the Houses of Parliament.
- Lunch: A pub near Westminster.
- Afternoon: Imperial War Museum. Explore exhibits about British conflicts from World War I to the present.
- Evening: A cocktail at Nightjar, followed by dinner.


**Prompt:** I'm traveling to London for next 3 days and I love history, cocktails, and some fashion.

---

**Agent Response:**

Okay, here's what I found regarding cocktail and fashion events during your trip:

**Cocktail Events:**

*   **Cocktails & Conversation at Polo in the Park:** June 8, 2025, at Hurlingham Park.
*   **Curry&Cocktail:** June 10, 2025, with The Penge Festival.

**Fashion Events:**

*   **Fashion District Festival:** June 3–8, 2025, in East London’s Spitalfields. It will feature sustainable, circular, and regenerative fashion.
*   **London Fashion Week:** June 14-18, 2025. However, the June 2025 edition of London Fashion Week has been cancelled.
*   **London Fashion Week Men's Spring/Summer 2026:** June 12 - 16, 2025

Given this information, you might consider attending the Fashion District Festival for some sustainable fashion.


**Prompt:** I'm traveling to London for next 3 days and I love history, cocktails, and some fashion.

---

**Agent Response:**

Okay, I've reviewed the itinerary and the events list.

The Fashion District Festival takes place from June 3-8, which is before your trip. London Fashion Week Men's Spring/Summer 2026 is from June 12-16, which is after your trip.

I will integrate the "Curry&Cocktail" event into your itinerary on June 10th.

**Final Itinerary:**

### Day 1 (June 9, 2025)

*   Morning: Tower of London. Explore the historic castle, see the Crown Jewels, and learn about its history.
*   Lunch: The Perkin Reveller near the Tower of London offers a modern British cuisine with historical charm.
*   Afternoon: Vintage Shopping at Brick Lane. Explore the vintage fashion scene at Brick Lane.
*   Evening: Cocktail Hour at The Churchill Bar & Terrace. Enjoy a historical theme and sophisticated cocktails. Followed by dinner at a nearby restaurant.

### Day 2 (June 10, 2025)

*   Morning: British Museum. Focus on historical exhibits like the Rosetta Stone and Elgin Marbles.
*   Lunch: Enjoy food near the British Museum.
*   Afternoon: Fashion History at the Victoria and Albert Museum. Explore fashion through the ages at the V&A.
*   Evening: **Curry&Cocktail with The Penge Festival.** Enjoy a unique event combining Curry and Cocktails.

### Day 3 (June 11, 2025)

*   Morning: Westminster Abbey & Houses of Parliament. Tour Westminster Abbey and visit the Houses of Parliament.
*   Lunch: A pub near Westminster.
*   Afternoon: Imperial War Museum. Explore exhibits about British conflicts from World War I to the present.
*   Evening: A cocktail at Nightjar, followed by dinner.


**Prompt:** I'm traveling to London for next 3 days and I love history, cocktails, and some fashion.

---

**Agent Response:**

Okay, I have your itinerary for London from June 9 to June 11, 2025, including historical sites, cocktail locations and vintage shopping in Brick Lane, plus the weather forecast. Here's a packing list tailored to your trip:

#### Clothing
- Comfortable walking shoes (essential for all the walking tours and museum visits)
- Comfortable socks
- Layers (t-shirts, long-sleeved shirts)
- Sweater or fleece jacket
- A light jacket or raincoat (London weather can be unpredictable)
- Jeans or comfortable pants
- Smart casual outfit for dinner at a nearby restaurant after The Churchill Bar & Terrace and dinner at Kaspar's at The Savoy.
- Cocktail attire for The Churchill Bar & Terrace, The Savoy's American Bar and Nightjar
- Outfit for vintage shopping in Brick Lane
- Optional: A hat and gloves if you feel cold

#### Toiletries
- Toothbrush, toothpaste, floss
- Shampoo, conditioner, body wash
- Deodorant
- Sunscreen (even on cloudy days)
- Any personal prescriptions
- Pain relievers
- Basic first-aid supplies

#### Electronics
- Phone and charger
- Portable charger (power bank) for long days of sightseeing
- UK plug adapter (if needed)
- Camera to capture the historical sites and fashion finds
- Optional: Tablet or e-reader

#### Documents
- Passport
- Travel insurance details
- Copies of important documents (stored separately)
- Tickets or booking confirmations for tours and events
- Credit cards and cash

#### Other Essentials
- Daypack for carrying essentials during the day
- Reusable water bottle
- Umbrella (London is known for rain)
- Small notebook and pen for jotting down notes
- Hand sanitizer
- Guidebook or map of London
- Snacks for long days of exploring
- Comfortable shoulder bag/ purse

Session state (per conversation):
  user_preference_temperature_unit: imperial
  weather_summary: OK. The weather in London for the next 3 days will be partly cloudy with average temperature of 60.7F, with a low of 50.8F and a high of 68.7F.

  latest_events: Okay, here's what I found regarding cocktail and fashion events during your trip:

**Cocktail Events:**

*   **Cocktails & Conversation at Polo in the Park:** June 8, 2025, at Hurlingham Park.
*   **Curry&Cocktail:** June 10, 2025, with The Penge Festival.

**Fashion Events:**

*   **Fashion District Festival:** June 3–8, 2025, in East London’s Spitalfields. It will feature sustainable, circular, and regenerative fashion.
*   **London Fashion Week:** June 14-18, 2025. However, the June 2025 edition of London Fashion Week has been cancelled.
*   **London Fashion Week Men's Spring/Summer 2026:** June 12 - 16, 2025

Given this information, you might consider attending the Fashion District Festival for some sustainable fashion.

  iti